In [105]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=V5XYMb6s7E1XMYY93rCgLAUApvyMGK&access_type=offline&code_challenge=DaC748rWYZFDN0wGZqKZk3ms2La2e9mvl8gG_YO8F9M&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)


session = GentropySession()

26/05/08 16:36:47 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@502f8a78{/SQL,null,AVAILABLE,@Spark}
26/05/08 16:36:47 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@659bf12d{/SQL/json,null,AVAILABLE,@Spark}
26/05/08 16:36:47 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@23c9ec70{/SQL/execution,null,AVAILABLE,@Spark}
26/05/08 16:36:47 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@dd20414{/SQL/execution/json,null,AVAILABLE,@Spark}
26/05/08 16:36:47 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@1fd5e652{/static/sql,null,AVAILABLE,@Spark}
26/05/08 16:36:47 WARN session: Consider rebuilding SparkSession to apply requested configuration: 'spark.driver.extraJavaOptions' has value '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-U

In [6]:
path_to_release_folder="gs://open-targets-data-releases/26.03/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

# Combining with FM

In [7]:
fm=session.spark.read.parquet(path_to_release_folder+"intermediate/l2g_feature_matrix/")

In [8]:
cs=sl.df.select("studyLocusId","studyId")
result_df = fm.join(cs, on="studyLocusId", how="left")

In [9]:
si_df=si.df.select("studyId","diseaseIds").dropDuplicates(["studyId"])
result_df = result_df.join(si_df, on="studyId", how="left")

In [10]:
result_df.count()

26/05/08 16:37:35 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=544; previousMaxLatencyMs=491; operationCount=144; context=gs://open-targets-data-releases/26.03/intermediate/l2g_feature_matrix/part-00017-2d393cda-895b-4041-9d29-4c48503fbe63-c000.snappy.parquet
26/05/08 16:37:35 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=607; previousMaxLatencyMs=544; operationCount=150; context=gs://open-targets-data-releases/26.03/intermediate/l2g_feature_matrix/part-00026-2d393cda-895b-4041-9d29-4c48503fbe63-c000.snappy.parquet
26/05/08 16:37:35 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=637; previousMaxLatencyMs=607; operationCount=162; context=gs://open-targets-data-releases/26.03/intermediate/l2g_feature_matrix/part-00053-2d393cda-895b-4041-9d29-4c48503fbe63-c000.snappy.parquet
26/05/08 16:37:40 INFO GhfsStor

58050449

In [11]:
fm.count()

58050449

In [12]:
sgl=session.spark.read.parquet("2603_EGL_0.75_otg_chembl.parquet")

In [13]:
sgl.count()

55003

In [14]:
sgl = sgl.select("targetId","diseaseId").withColumnRenamed("targetId", "geneId_sgl")

In [15]:
joined_df = result_df.join(sgl, (f.array_contains(result_df["diseaseIds"], sgl["diseaseId"]))&(result_df["geneId"]==sgl["geneId_sgl"]),how="left")

In [16]:
df=joined_df.filter(f.col("diseaseId").isNotNull())
df=df.drop_duplicates(["studyLocusId","geneId","diseaseIds"])

In [17]:
df=df.cache()
df.count()

26/05/08 16:41:31 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=6102; previousMaxLatencyMs=4719; operationCount=1048860; context=gs://open-targets-data-releases/26.03/intermediate/l2g_feature_matrix/part-00131-2d393cda-895b-4041-9d29-4c48503fbe63-c000.snappy.parquet


27364

In [18]:
df.drop_duplicates(["geneId","diseaseIds"]).count()

4026

In [19]:
study_locus_ids = df.select("studyLocusId")
study_locus_id_list = [row.studyLocusId for row in study_locus_ids.collect()]

In [20]:
len(study_locus_id_list)

27364

In [21]:
result_df.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- credibleSetConfidence: float (nullable = true)
 |-- distanceFootprintMean: float (nullable = true)
 |-- distanceFootprintMeanNeighbourhood: float (nullable = true)
 |-- distanceSentinelFootprint: float (nullable = true)
 |-- distanceSentinelFootprintNeighbourhood: float (nullable = true)
 |-- distanceSentinelTss: float (nullable = true)
 |-- distanceSentinelTssNeighbourhood: float (nullable = true)
 |-- distanceTssMean: float (nullable = true)
 |-- distanceTssMeanNeighbourhood: float (nullable = true)
 |-- e2gMean: float (nullable = true)
 |-- e2gMeanNeighbourhood: float (nullable = true)
 |-- eQtlColocClppMaximum: float (nullable = true)
 |-- eQtlColocClppMaximumNeighbourhood: float (nullable = true)
 |-- eQtlColocH4Maximum: float (nullable = true)
 |-- eQtlColocH4MaximumNeighbourhood: float (nullable = true)
 |-- geneCount500kb: double (nullable = true)
 

In [22]:
fm1=result_df.filter(f.col("studyLocusId").isin(study_locus_id_list))

In [23]:
fm1.count()

1123880

In [24]:
df.count()

27364

In [25]:
fm1 = fm1.join(
    df.select("studyLocusId", "geneId").withColumn("GSP", f.lit(1)),
    on=["studyLocusId", "geneId"],
    how="left"
).withColumn(
    "GSP",
    f.when(f.col("GSP").isNotNull(), 1).otherwise(0)
).cache()

# Show the result
fm1.show(1)

+--------------------+---------------+------------+---------------------+---------------------+----------------------------------+-------------------------+--------------------------------------+-------------------+--------------------------------+---------------+----------------------------+-------+--------------------+--------------------+---------------------------------+------------------+-------------------------------+--------------+--------------------+---------------------------------+------------------+-------------------------------+---------------------+--------------------+---------------------------------+------------------+-------------------------------+----------+-----------------------+-------+--------------------+---------------+-------------+---+
|        studyLocusId|         geneId|     studyId|credibleSetConfidence|distanceFootprintMean|distanceFootprintMeanNeighbourhood|distanceSentinelFootprint|distanceSentinelFootprintNeighbourhood|distanceSentinelTss|distanceS

In [26]:
fm1.count()

1123880

In [27]:
fm1.filter(f.col("GSP") == 1).count()

27364

In [28]:
fm1.filter((f.col("GSP") == 1)&(f.col("isProteinCoding")==1)).count()

27293

In [29]:
fm1.write.mode("overwrite").parquet("unfiltered_FM_with_GSP.parquet")

# Making training set - step 1

In [53]:
fm=session.spark.read.parquet("unfiltered_FM_with_GSP.parquet")

In [54]:
fm.printSchema()

root
 |-- studyLocusId: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- credibleSetConfidence: float (nullable = true)
 |-- distanceFootprintMean: float (nullable = true)
 |-- distanceFootprintMeanNeighbourhood: float (nullable = true)
 |-- distanceSentinelFootprint: float (nullable = true)
 |-- distanceSentinelFootprintNeighbourhood: float (nullable = true)
 |-- distanceSentinelTss: float (nullable = true)
 |-- distanceSentinelTssNeighbourhood: float (nullable = true)
 |-- distanceTssMean: float (nullable = true)
 |-- distanceTssMeanNeighbourhood: float (nullable = true)
 |-- e2gMean: float (nullable = true)
 |-- e2gMeanNeighbourhood: float (nullable = true)
 |-- eQtlColocClppMaximum: float (nullable = true)
 |-- eQtlColocClppMaximumNeighbourhood: float (nullable = true)
 |-- eQtlColocH4Maximum: float (nullable = true)
 |-- eQtlColocH4MaximumNeighbourhood: float (nullable = true)
 |-- geneCount500kb: double (nullable = true)
 

In [55]:
fm.count()

1123880

In [56]:
fm.filter(f.col("GSP") == 1).count()

27364

# Replication criteria

In [57]:
si_df = si.df.select("studyId", "diseaseIds", "geneId", "biosampleId", "pubmedId", "cohorts", "ldPopulationStructure")
result_df = (
    sl.df.select("studyId", "variantId", "studyType", "studyLocusId").join(si_df, on="studyId", how="left").cache()
)


In [58]:
gwas = result_df.filter(f.col("studyType") == "gwas").cache()
gwas.count()


1446959

In [59]:
gwas = gwas.withColumn("diseaseId", f.explode(f.col("diseaseIds"))).cache()
gwas.count()


1509461

In [60]:
df = gwas.select("variantId", "diseaseId", "cohorts", "pubmedId", "ldPopulationStructure")
df.count()


1509461

In [61]:
df = df.dropDuplicates()
df.count()


1270069

In [62]:
grouped = df.groupBy("variantId", "diseaseId").agg(f.count("*").alias("count"))
filtered = grouped.filter(f.col("count") >= 2)
result = gwas.join(filtered, on=["variantId", "diseaseId"], how="inner").select("studyLocusId").distinct()
result.count()


654390

In [63]:
repl_cs=result.select("studyLocusId").distinct().cache()
repl_cs.count()

654390

In [64]:
repl_cs.count()

654390

In [65]:
fm=fm.join(repl_cs, on="studyLocusId", how="inner").cache()
fm.count()

654512

In [66]:
fm.filter(f.col("GSP") == 1).count()

15999

# Max two GSP per CS

In [67]:
grouped = fm.filter(f.col("GSP")==1).groupBy("studyLocusId").agg(f.count("*").alias("count"))
filtered = grouped.filter(f.col("count")>2).select("studyLocusId").cache()
filtered.count()

196

In [68]:
grouped = fm.filter(f.col("GSP")==1).groupBy("studyLocusId").agg(f.count("*").alias("count"))
filtered = grouped.filter(f.col("count")>0).filter(f.col("count")<=2).select("studyLocusId").cache()
filtered.count()

14711

In [69]:
filtered.show(1)

+--------------------+
|        studyLocusId|
+--------------------+
|02af44369530b53a5...|
+--------------------+
only showing top 1 row



In [70]:
fm=fm.join(filtered, on="studyLocusId", how="inner").cache()
fm.count()

645718

In [71]:
fm.filter((f.col("GSP")==1)).count()

15354

# String filter

In [72]:
inter=session.spark.read.parquet(path_to_release_folder+"/output/interaction/")
inter.show(1)

+--------------+---------------+------+------------------+---------------+------+------------------+--------------------+--------------------+-----+-------+
|sourceDatabase|        targetA|  intA|intABiologicalRole|        targetB|  intB|intBBiologicalRole|            speciesA|            speciesB|count|scoring|
+--------------+---------------+------+------------------+---------------+------+------------------+--------------------+--------------------+-----+-------+
|        intact|ENSG00000173039|Q04206|     enzyme target|ENSG00000068903|Q8IXJ6|            enzyme|{human, Homo sapi...|{human, Homo sapi...|    1|   0.54|
+--------------+---------------+------+------------------+---------------+------+------------------+--------------------+--------------------+-----+-------+
only showing top 1 row



In [73]:
inter.groupBy("sourceDatabase").agg(f.count("*").alias("count")).show()

+--------------+--------+
|sourceDatabase|   count|
+--------------+--------+
|        intact| 1311132|
|        signor|   39992|
|      reactome|   56191|
|        string|13210738|
+--------------+--------+



In [74]:
inter=inter.filter((f.col("sourceDatabase")=="string")&(f.col("scoring")>=0.75)&(~(f.col("targetA")==f.col("targetB")))).select("targetA","targetB").distinct().cache()
#inter=inter.filter(~(f.col("targetA")==f.col("targetB"))).select("targetA","targetB").distinct().cache()
inter.count()

378436

In [75]:
inter.show(1)

+---------------+---------------+
|        targetA|        targetB|
+---------------+---------------+
|ENSG00000067646|ENSG00000092377|
+---------------+---------------+
only showing top 1 row



In [76]:
fm1=fm.filter(f.col("GSP")==1).select("geneId","studyLocusId").cache()
fm1.count()

15354

In [77]:
fm1.select("geneId").distinct().count()

517

In [78]:
result = fm1.join(
    inter,
    fm1["geneId"] == inter["targetA"],
    how="left"
).select(
    "geneId",
    "studyLocusId",
    "targetB"
).cache()

result.count()

661841

In [79]:
result.filter(f.col("targetB").isNull()).count()

191

In [80]:
result.filter(f.col("targetB")==f.col("geneId")).count()

0

In [81]:
result.select(f.col("geneId")).distinct().count()

517

In [82]:
result=result.filter(f.col("targetB").isNotNull()).select("targetB","studyLocusId").withColumnRenamed("targetB","geneId").cache()
result.count()

661650

In [83]:
fm_s=fm.filter(f.col("GSP")==0).select("geneId","studyLocusId").cache()
fm_s.count()

630364

In [84]:
fm_s=fm_s.union(result.select("geneId","studyLocusId")).cache()
fm_s.count()

1292014

In [85]:
grouped = fm_s.groupBy("geneId", "studyLocusId").agg(f.count("*").alias("count"))
filtered = grouped.filter(f.col("count") >= 2).select("geneId", "studyLocusId").cache()
filtered.count()

16965

In [86]:
fm.filter(f.col("GSP")==1).count()

15354

In [87]:
fm.count()

645718

In [88]:
fm_no_inter=fm.join(filtered, on=["geneId", "studyLocusId"], how="anti").cache()
fm_no_inter.count()

638307

In [89]:
fm_no_inter.filter(f.col("GSP")==1).count()

15354

In [90]:
fm_no_inter.filter(f.col("GSP")==1).select(f.col("geneId")).distinct().count()

517

In [91]:
fm_no_inter.filter(f.col("GSP")==1).select("geneId","diseaseIds").distinct().count()

1755

In [92]:
fm_no_inter.write.mode("overwrite").parquet("preset_no_interaction.parquet")

# Distance criteria

In [93]:
fm_no_inter=session.spark.read.parquet("preset_no_interaction.parquet")

In [94]:
fm_no_inter.printSchema()

root
 |-- geneId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- credibleSetConfidence: float (nullable = true)
 |-- distanceFootprintMean: float (nullable = true)
 |-- distanceFootprintMeanNeighbourhood: float (nullable = true)
 |-- distanceSentinelFootprint: float (nullable = true)
 |-- distanceSentinelFootprintNeighbourhood: float (nullable = true)
 |-- distanceSentinelTss: float (nullable = true)
 |-- distanceSentinelTssNeighbourhood: float (nullable = true)
 |-- distanceTssMean: float (nullable = true)
 |-- distanceTssMeanNeighbourhood: float (nullable = true)
 |-- e2gMean: float (nullable = true)
 |-- e2gMeanNeighbourhood: float (nullable = true)
 |-- eQtlColocClppMaximum: float (nullable = true)
 |-- eQtlColocClppMaximumNeighbourhood: float (nullable = true)
 |-- eQtlColocH4Maximum: float (nullable = true)
 |-- eQtlColocH4MaximumNeighbourhood: float (nullable = true)
 |-- geneCount500kb: double (nullable = true)
 

In [95]:
fm_no_inter.filter(((f.col("GSP")==1)&(f.col("distanceSentinelFootprint")==0))).count()

453

In [96]:
fm_0=fm_no_inter.filter(~((f.col("GSP")==1)&(f.col("distanceSentinelFootprint")==0))).cache()
fm_0.count()

637854

In [97]:
fm_0.filter(f.col("GSP")==1).select("geneId","diseaseIds").distinct().count()

1713

In [98]:
fm_0.filter(f.col("GSP")==1).count()

14901

In [99]:
fm_0.filter(f.col("GSP")==0).count()

622953

In [100]:
fm_0.filter((f.col("GSP")==1)&(f.col("isProteinCoding")==1)).count()

14881

In [101]:
fm_0.filter((f.col("GSP")==1)&(f.col("isProteinCoding")==0)).count()

20

In [102]:
fm_0.write.mode("overwrite").parquet("preset_distance_filtered_all_genes.parquet")

# Protein-coding only genes

In [103]:
fm=session.spark.read.parquet("preset_distance_filtered_all_genes.parquet")

In [104]:
fm_0=fm.filter(f.col("isProteinCoding")==1).cache()
fm_0.count()

236144

In [105]:
grouped = fm_0.filter(f.col("GSP")==1).groupBy("studyLocusId").agg(f.count("*").alias("count"))
filtered = grouped.filter(f.col("count")>0).filter(f.col("count")<=2).select("studyLocusId").cache()
filtered.count()

14343

In [106]:
filtered.show(1)

+--------------------+
|        studyLocusId|
+--------------------+
|02af44369530b53a5...|
+--------------------+
only showing top 1 row



In [107]:
fm_0=fm_0.join(filtered, on="studyLocusId", how="inner").cache()
fm_0.count()

227259

In [108]:
fm_0.write.mode("overwrite").parquet("prefinal_training_set_protein_coding_only_genes.parquet")

# Patching - removing duplicates

In [109]:
fm=session.spark.read.parquet("prefinal_training_set_protein_coding_only_genes.parquet")
fm.count()

227259

In [110]:
fm=fm.join(sl.df.select("studyLocusId","variantId"), on="studyLocusId", how="left").cache()
fm.count()

227259

In [111]:
n1 = fm.count()
tmp1 = fm.filter(f.col("GSP") == 1)
colnms = ["geneId", "diseaseIds", "variantId", "vepMaximum", "vepMean"]
colnms_to_round = [
    "eQtlColocClppMaximum", "pQtlColocClppMaximum", "sQtlColocClppMaximum",
    "eQtlColocH4Maximum", "pQtlColocH4Maximum", "sQtlColocH4Maximum"
]

In [112]:
# Step 4: Round specified columns to 2 decimal places
for col in colnms_to_round:
    tmp1 = tmp1.withColumn(col, f.round(f.col(col), 2))

In [113]:
# Step 5: Remove duplicates based on specified columns
tmp1 = tmp1.dropDuplicates(colnms + colnms_to_round)

# Step 6: Get unique studyLocusId values to keep
cs_to_keep = tmp1.select("studyLocusId").distinct()

# Step 7: Filter the original DataFrame to keep only rows with studyLocusId in cs_to_keep
fm_0 = fm.join(cs_to_keep, on="studyLocusId", how="inner")

In [114]:
# Step 8: Get the new row count
n2 = fm_0.count()

# Step 9: Print the reduction percentage
reduction_percentage = round(((n1 - n2) / n1) * 100, 2)
print(f"Reduced by: {reduction_percentage}%")

Reduced by: 11.76%


In [115]:
fm_0.count()

200536

In [116]:
fm_0.filter(f.col("GSP")==1).count()

13060

In [117]:
fm_0.filter(f.col("GSP")==1).select("geneId","diseaseIds").distinct().count()

1711

In [118]:
fm_0.filter(f.col("GSP")==1).select("geneId").distinct().count()

505

In [119]:
fm_0.filter(f.col("GSP")==1).count()

13060

In [120]:
fm_0.filter(f.col("GSP")==0).count()

187476

In [121]:
fm_0.write.parquet("2603_training_set_full_fm.parquet",mode="overwrite")

# Convert to JSON

In [122]:
fm_0=session.spark.read.parquet("2603_training_set_full_fm.parquet")

In [123]:
x=fm_0.select("studyLocusId","geneId","GSP","diseaseIds","variantId","studyId")

In [124]:
x.count()

200536

In [125]:
x.show(1)

+--------------------+---------------+---+-------------+--------------+------------+
|        studyLocusId|         geneId|GSP|   diseaseIds|     variantId|     studyId|
+--------------------+---------------+---+-------------+--------------+------------+
|015ac1639002504fe...|ENSG00000122008|  0|[EFO_0004574]|5_75355259_A_G|GCST90500015|
+--------------------+---------------+---+-------------+--------------+------------+
only showing top 1 row



In [126]:
result_df = x.withColumn(
    "goldStandardSet",
    f.when(f.col("GSP") == 1, "positive").otherwise("negative")
).drop("GSP").cache()
result_df.count()

200536

In [127]:
result_df.show(3)

+--------------------+---------------+-------------+--------------+------------+---------------+
|        studyLocusId|         geneId|   diseaseIds|     variantId|     studyId|goldStandardSet|
+--------------------+---------------+-------------+--------------+------------+---------------+
|015ac1639002504fe...|ENSG00000122008|[EFO_0004574]|5_75355259_A_G|GCST90500015|       negative|
|015ac1639002504fe...|ENSG00000152359|[EFO_0004574]|5_75355259_A_G|GCST90500015|       negative|
|015ac1639002504fe...|ENSG00000113161|[EFO_0004574]|5_75355259_A_G|GCST90500015|       positive|
+--------------------+---------------+-------------+--------------+------------+---------------+
only showing top 3 rows



In [128]:
result_df.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 13060|
|       negative|187476|
+---------------+------+



In [129]:
result_df.write.json("2603_training_set_full_fm.json",mode="overwrite")